In [10]:
import pandas as pd
from surprise import Dataset, Reader,SVD
from surprise.model_selection import train_test_split

In [31]:
ratings = ratings = pd.read_csv( "../data/ratings.csv")
movies = pd.read_csv("../data/movies.csv")
reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(
    ratings[['userId', 'movieId', 'rating']],
    reader
)

In [9]:
trainset , testset = train_test_split(data,test_size =0.2, random_state =42)

In [13]:
model = SVD()
model.fit(trainset)

In [19]:
prediction=  model.predict(uid=1, iid=50)
print(prediction.est)

4.854698813066769


In [20]:
ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB


In [21]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [22]:
ratings['date'] = pd.to_datetime(
    ratings['timestamp'],
    unit='s'
)

ratings[['timestamp', 'date']].head()

,timestamp,date
0,964982703,2000-07-30 18:45:03
1,964981247,2000-07-30 18:20:47
2,964982224,2000-07-30 18:37:04
3,964983815,2000-07-30 19:03:35
4,964982931,2000-07-30 18:48:51


In [23]:
from surprise import accuracy

In [24]:
predictions = model.test(testset)

In [25]:
accuracy.rmse(predictions)

RMSE: 0.8781


0.8781055490057458

In [28]:
rated_movies = set(
    ratings[
        ratings["userId"] == 1
    ]["movieId"]
)

In [32]:
all_movies = set(
    movies["movieId"]
)

In [33]:
unseen_movies = all_movies - rated_movies

len(unseen_movies)

9510

In [34]:
predictions = []

for movie_id in unseen_movies:
    
    pred = model.predict(
        uid=1,
        iid=movie_id
    )
    
    predictions.append(
        (movie_id, pred.est)
    )

In [35]:
predictions = sorted(
    predictions,
    key=lambda x: x[1],
    reverse=True
)

In [37]:
top_10 = predictions[:10]
top_10

[(904, 5.0),
 (912, 5.0),
 (1201, 5.0),
 (1217, 5.0),
 (1719, 5.0),
 (1953, 5.0),
 (3275, 5.0),
 (27773, 5.0),
 (8368, 4.999816558826942),
 (112852, 4.999726273149753)]

In [45]:
top_movies = pd.DataFrame(
    top_10,
    columns=["movieId", "predicted_rating"]
)

top_movies = top_movies.merge(
    movies,
    on="movieId"
)

top_movies[
    ["title", "predicted_rating", "genres"]
]

,title,predicted_rating,genres
0,Rear Window (1954),5.000000,Mystery|Thriller
1,Casablanca (1942),5.000000,Drama|Romance
2,"Good, the Bad and the Ugly, The (Buono, il bru...",5.000000,Action|Adventure|Western
3,Ran (1985),5.000000,Drama|War
4,"Sweet Hereafter, The (1997)",5.000000,Drama
5,"French Connection, The (1971)",5.000000,Action|Crime|Thriller
6,"Boondock Saints, The (2000)",5.000000,Action|Crime|Drama|Thriller
7,Old Boy (2003),5.000000,Mystery|Thriller
8,Harry Potter and the Prisoner of Azkaban (2004),4.999817,Adventure|Fantasy|IMAX
9,Guardians of the Galaxy (2014),4.999726,Action|Adventure|Sci-Fi


In [39]:
movies[
    movies["title"].isin([
        "Rear Window (1954)",
        "Casablanca (1942)",
        "Ran (1985)",
        "Sweet Hereafter, The (1997)"
    ])
][["title","genres"]]

,title,genres
686,Rear Window (1954),Mystery|Thriller
694,Casablanca (1942),Drama|Romance
918,Ran (1985),Drama|War
1290,"Sweet Hereafter, The (1997)",Drama


In [40]:
movies[
    movies["title"].str.contains(
        "Good, the Bad and the Ugly",
        case=False,
        na=False
    )
][["title","genres"]]

,title,genres
903,"Good, the Bad and the Ugly, The (Buono, il bru...",Action|Adventure|Western


In [41]:
user1_history = ratings[
    ratings["userId"] == 1
]

user1_movies = user1_history.merge(
    movies,
    on="movieId"
)

user1_movies[[
    "title",
    "rating",
    "genres"
]].sort_values(
    by="rating",
    ascending=False
)

,title,rating,genres
231,M*A*S*H (a.k.a. MASH) (1970),5.0,Comedy|Drama|War
185,Excalibur (1981),5.0,Adventure|Fantasy
89,Indiana Jones and the Last Crusade (1989),5.0,Action|Adventure
90,Pink Floyd: The Wall (1982),5.0,Drama|Musical
190,From Russia with Love (1963),5.0,Action|Adventure|Thriller
...,...,...,...
170,"Mummy, The (1999)",2.0,Action|Adventure|Comedy|Fantasy|Horror|Thriller
143,Toys (1992),2.0,Comedy|Fantasy
148,I Still Know What You Did Last Summer (1998),2.0,Horror|Mystery|Thriller
152,Psycho (1998),2.0,Crime|Horror|Thriller


In [43]:
user1_movies.head(10)

,userId,movieId,rating,timestamp,title,genres
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,1,3,4.0,964981247,Grumpier Old Men (1995),Comedy|Romance
2,1,6,4.0,964982224,Heat (1995),Action|Crime|Thriller
3,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,5.0,964982931,"Usual Suspects, The (1995)",Crime|Mystery|Thriller
5,1,70,3.0,964982400,From Dusk Till Dawn (1996),Action|Comedy|Horror|Thriller
6,1,101,5.0,964980868,Bottle Rocket (1996),Adventure|Comedy|Crime|Romance
7,1,110,4.0,964982176,Braveheart (1995),Action|Drama|War
8,1,151,5.0,964984041,Rob Roy (1995),Action|Drama|Romance|War
9,1,157,5.0,964984100,Canadian Bacon (1995),Comedy|War
